In [1]:
import sys
sys.path.append("..")
import os
import numpy as np
import pandas as pd
from datetime import datetime
from src.metrics import percentile_quadrant, mann_whitney_with_effect_size
from src.db import query_cached, append_note
from src.cleaning import detect_contaminated_devices, filter_non_laptop_devices, normalize_laptop_weight

df_analysis = pd.read_csv("../data/processed/analysis_base.csv")

In [2]:
### [Giai đoạn 5 - Trang 1] Overview stats cho Manager
overview_query = """
SELECT
    COUNT(DISTINCT user_psuedo_id) AS total_active_users,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(*) AS total_events
FROM laplaptech.user_event_tracking
"""
df_overview = query_cached(overview_query, cache_dir="../data/raw", force_refresh=False)
display(df_overview)

# Phân bố "người dùng" theo phân khúc máy: dùng interest_score/pageview_count (mức độ chú ý
# thực tế của người dùng), KHÔNG chỉ đếm số lượng laptop trong từng phân khúc.
def segment_label(row):
    if row["is_gaming_laptop"] == True:
        return "Gaming"
    if row["is_workstation"] == True:
        return "Workstation"
    return "Office"

df_analysis["segment"] = df_analysis.apply(segment_label, axis=1)
df_segment_attention = df_analysis.groupby("segment").agg(
    laptop_count=("id", "count"),
    total_pageview=("pageview_count", "sum"),
    total_interest_score=("interest_score", "sum"),
).reset_index()
display(df_segment_attention)

df_overview.to_csv("../data/processed/dashboard_overview_kpi.csv", index=False)
print("Đã lưu: dashboard_overview_kpi.csv")
print("Top laptop dùng trực tiếp dashboard_laptop.csv, không cần file riêng.")

,total_active_users,total_sessions,total_events
0,35291,88323,781680


,segment,laptop_count,total_pageview,total_interest_score
0,Gaming,47,30667,46503
1,Office,84,72287,113933
2,Workstation,2,1819,2699


Đã lưu: dashboard_overview_kpi.csv
Top laptop dùng trực tiếp dashboard_laptop.csv, không cần file riêng.


In [3]:
### [Giai đoạn 5 - Trang 2] CPU/GPU battery ratio — flag rõ độ thưa, dùng battery_density thay thế
cpu_gpu_query = """
SELECT
    c.name AS cpu_name,
    b.geekbench_6_cpu_multi_core_plugged_in,
    b.geekbench_6_cpu_multi_core_battery,
    b.laptop_model_id
FROM laplaptech.laptop_benchmark_result b
INNER JOIN laplaptech.laptop_model m ON b.laptop_model_id = m.id
INNER JOIN laplaptech.cpu_model c ON m.cpu_model_id = c.id
WHERE b.geekbench_6_cpu_multi_core_plugged_in IS NOT NULL
  AND b.geekbench_6_cpu_multi_core_battery IS NOT NULL
"""
df_cpu_gpu = query_cached(cpu_gpu_query, cache_dir="../data/raw", force_refresh=False)
df_cpu_gpu["cpu_battery_ratio"] = (
    df_cpu_gpu["geekbench_6_cpu_multi_core_battery"] / df_cpu_gpu["geekbench_6_cpu_multi_core_plugged_in"]
)
print(f"Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: {len(df_cpu_gpu)}")

# battery_density theo hãng CPU — chỉ số thay thế có độ phủ tốt hơn (population đã lọc sạch)
df_laptop_raw = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")
df_cpu_names = query_cached("SELECT id, name FROM laplaptech.cpu_model", cache_dir="../data/raw")
df_laptop_clean = filter_non_laptop_devices(df_laptop_raw)
df_laptop_clean = df_laptop_clean.merge(df_cpu_names, left_on="cpu_model_id", right_on="id",
                                          suffixes=("", "_cpu"))
df_laptop_clean["laptop_weight_g"] = normalize_laptop_weight(df_laptop_clean)
df_laptop_clean["battery_density"] = pd.to_numeric(
    df_laptop_clean["battery_capacity_whr"], errors="coerce"
) / df_laptop_clean["laptop_weight_g"].replace(0, pd.NA)

df_density_by_cpu = df_laptop_clean.groupby("name_cpu")["battery_density"].agg(
    ["count", "mean", "median"]
).reset_index().sort_values("count", ascending=False)
display(df_density_by_cpu)
df_density_by_cpu.to_csv("../data/processed/dashboard_battery_density_by_cpu.csv", index=False)

note_page2 = f"""
## [Giai đoạn 5 - Trang 2] Ghi chú độ khả dụng dữ liệu

CPU_Battery_Ratio: chỉ {len(df_cpu_gpu)} dòng đủ dữ liệu Geekbench -> KHÔNG dùng làm biểu đồ
chính (không xuất file). Thay thế: dashboard_battery_density_by_cpu.csv, độ phủ tốt
hơn nhiều, dùng cùng Scatter Trọng lượng vs Battery Density đã có trong analysis_base.csv.
"""
append_note(note_page2)

Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: 3


,name_cpu,count,mean,median
35,Intel Core Ultra 7 258V,9,0.054076,0.056062
7,AMD Ryzen AI 7 350,6,0.051640,0.051000
30,Intel Core Ultra 7 155H,5,0.045290,0.048993
11,AMD Ryzen AI 9 HX 370,5,0.052464,0.051714
29,Intel Core Ultra 5 226V,4,0.050685,0.051546
...,...,...,...,...
58,Intel Core i9-14900H,1,0.032143,0.032143
63,Qualcomm Snapdragon X2 Elite Extreme X2E 96 100,1,0.058333,0.058333
61,Qualcomm Snapdragon X Plus X1P42100,1,0.038514,0.038514
64,Qualcomm Snapdragon X2 Elite X2E 88 100,1,0.070707,0.070707


In [4]:
### [Giai đoạn 5 - Trang 3] Điều tra hiện tượng không đơn điệu giảm + kết luận CHÍNH THỨC
# (Lịch sử đầy đủ của quá trình điều tra — bao gồm bước từng overclaim rồi tự đính chính —
#  đã được lưu vĩnh viễn trong project_notes.md từ các lần chạy trước. Cell này chỉ giữ lại
#  logic tính toán và kết luận ĐÃ ĐÚNG, tránh lặp lại việc ghi đè note sai mỗi lần re-run.)

search_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking WHERE event_name = 'search_for_device'",
    cache_dir="../data/raw", force_refresh=False)["session_id"])
detail_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'",
    cache_dir="../data/raw", force_refresh=False)["session_id"])
browse_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'load_more_device_home' "
    "   OR (event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'Home')",
    cache_dir="../data/raw", force_refresh=False)["session_id"])

detail_without_search = detail_set - search_set
browse_and_no_search = detail_without_search & browse_set
pct_no_search = len(detail_without_search) / len(detail_set)
pct_confirmed = len(browse_and_no_search) / len(detail_without_search)

print(f"Xem chi tiết không qua tìm kiếm: {len(detail_without_search)} ({pct_no_search:.1%})")
print(f"  Trong đó xác nhận có duyệt qua Home/load_more: {pct_confirmed:.1%}")

funnel_final_note = f"""
## [Giai đoạn 5 - Trang 3] Kết luận chính thức về hiện tượng "Xem chi tiết" > "Tìm kiếm"

{len(detail_without_search)} session ({pct_no_search:.1%}) xem chi tiết KHÔNG qua tìm kiếm trước.
Trong nhóm này, {pct_confirmed:.1%} có bằng chứng đã duyệt qua Home/load_more_device_home —
MỘT PHẦN nguyên nhân, không phải toàn bộ (phần còn lại chưa xác định rõ nguồn gốc).

**Kết luận:** "Tìm kiếm -> Xem chi tiết -> Chọn So sánh" KHÔNG phải phễu chuyển đổi tuần tự bắt
buộc. Quyết định: Trang 3 dùng Bar chart "Mức độ tham gia theo từng hành động" thay vì Funnel
visual chuẩn (vốn ngụ ý quan hệ tập con giảm dần), kèm chú thích đúng mức độ bằng chứng.
"""
append_note(funnel_final_note)

note_page3 = """
## [Giai đoạn 5 - Trang 3] Ghi chú công thức funnel

Đếm số session PHÂN BIỆT có event tương ứng, không bắt buộc thứ tự thời gian trong cùng session.
Endpoint đổi từ "click video review" (kế hoạch gốc) sang "select_device_for_comparison", do
không có dữ liệu tracking cho hành vi xem video (xác nhận ở Giai đoạn 3).
"""
append_note(note_page3)

Xem chi tiết không qua tìm kiếm: 16394 (36.7%)
  Trong đó xác nhận có duyệt qua Home/load_more: 53.9%


In [5]:
### [Giai đoạn 5 - Data Quality Monitor] Snapshot % null + anomaly theo thời gian
snapshot_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_benchmark_raw = query_cached("SELECT * FROM laplaptech.laptop_benchmark_result", cache_dir="../data/raw")
# Tự query lại laptop_model tại đây (không dựa vào biến df_laptop_raw từ cell Trang 2)
# để cell này chạy độc lập được, đúng nguyên tắc reproducibility.
df_laptop_for_quality = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")
df_merged_quality = df_laptop_for_quality.merge(
    df_benchmark_raw, left_on="id", right_on="laptop_model_id", how="left", suffixes=("", "_bm")
)

cols_to_check = ["battery_capacity_whr", "laptop_weight", "cpu_tdp", "gpu_tdp",
                  "office_battery_result_minutes", "geekbench_6_cpu_multi_core_plugged_in"]
quality_rows = [
    {"snapshot_time": snapshot_time, "column": col,
     "null_pct": round(df_merged_quality[col].isna().mean(), 4)}
    for col in cols_to_check if col in df_merged_quality.columns
]
df_quality_snapshot = pd.DataFrame(quality_rows)
df_quality_snapshot["contaminated_device_count"] = len(detect_contaminated_devices(df_laptop_for_quality))
display(df_quality_snapshot)

quality_path = "../data/processed/dashboard_data_quality_history.csv"
# Tránh tích lũy nhiều snapshot trùng NGÀY khi chạy lại nhiều lần trong cùng 1 ngày để test
today_str = datetime.now().strftime("%Y-%m-%d")
if os.path.exists(quality_path):
    df_history = pd.read_csv(quality_path)
    df_history = df_history[~df_history["snapshot_time"].str.startswith(today_str)]
    df_combined = pd.concat([df_history, df_quality_snapshot], ignore_index=True)
else:
    df_combined = df_quality_snapshot
df_combined.to_csv(quality_path, index=False)
print(f"Đã cập nhật snapshot (thay thế snapshot cùng ngày nếu có) vào {quality_path}")

,snapshot_time,column,null_pct,contaminated_device_count
0,2026-10-17 20:33:30,battery_capacity_whr,0.0314,26
1,2026-10-17 20:33:30,laptop_weight,0.0314,26
2,2026-10-17 20:33:30,cpu_tdp,0.8868,26
3,2026-10-17 20:33:30,gpu_tdp,0.7862,26
4,2026-10-17 20:33:30,office_battery_result_minutes,0.9811,26
5,2026-10-17 20:33:30,geekbench_6_cpu_multi_core_plugged_in,0.9937,26


Đã cập nhật snapshot (thay thế snapshot cùng ngày nếu có) vào ../data/processed/dashboard_data_quality_history.csv


In [6]:
### [Giai đoạn 5] Data Dictionary cho Dashboard + quyết định refresh/schema
dashboard_dict = """
## [Giai đoạn 5] Data Dictionary cho Dashboard

| Metric hiển thị | Nguồn file | Công thức/Định nghĩa |
|---|---|---|
| Total Active Users | dashboard_overview_kpi.csv | COUNT(DISTINCT user_psuedo_id) |
| Total Sessions | dashboard_overview_kpi.csv | COUNT(DISTINCT session_id) |
| Interest Score | dashboard_laptop.csv (cột interest_score) | 1*pageview(DeviceDetail) + 2*select_device_for_comparison |
| Battery Density | dashboard_laptop.csv (cột density_wh_kg) | battery_wh / weight_kg (Wh/kg) |
| Golden Ratio Group | dashboard_laptop.csv (cột golden_group) | Top 25% battery_density + top 25% nhẹ nhất |
| Discovery Path | dashboard_discovery_path.csv | Tỷ trọng session theo nguồn vào trang chi tiết (Search-led/Browse-confirmed/Unknown) |
| Data Quality % Null | dashboard_data_quality_history.csv | Tỷ lệ NULL theo cột, 1 snapshot/ngày |

## [Giai đoạn 5] Quyết định Data Refresh & Schema Change

Refresh: chạy lại notebook 01->05, sau đó Refresh All trong Power BI. Không có lịch tự động.
Schema đổi: get_schema() ở notebook 01 sẽ báo lỗi sớm; detect_contaminated_devices() tự thích
ứng dữ liệu mới, không hardcode ID.
"""
append_note(dashboard_dict)

In [7]:
### [Giai đoạn 5] Tổng kết
stage5_summary = """
## [Giai đoạn 5] Tổng kết

Đã chuẩn bị đầy đủ dữ liệu nguồn cho Power BI: dashboard_overview_kpi.csv,
dashboard_battery_density_by_cpu.csv (dùng chính), dashboard_data_quality_history.csv.

Phát hiện quan trọng trong quá trình chuẩn bị: "funnel" Tìm kiếm->Xem chi tiết->Chọn So sánh
KHÔNG phải phễu tuần tự (36.7% xem chi tiết không qua tìm kiếm), đã đổi cách trình bày trang 3
từ Funnel visual sang Bar chart "Mức độ tham gia", tránh gây hiểu lầm.

Output đã sẵn sàng để dựng laplaptech_dashboard.pbix (4 trang: Overview, Hardware Benchmark
Matrix, User Engagement, Data Quality Monitor). Sẵn sàng hoàn tất Giai đoạn 6 (6.3 One-pager,
6.4 Publish Dashboard) sau khi hoàn thành thao tác thủ công trên Power BI.
"""
append_note(stage5_summary)

In [8]:
# ---- 1. Đơn vị hiển thị -------------------------------------------------
df = pd.read_csv("../data/processed/analysis_base.csv")
df["weight_kg"] = df["laptop_weight_g"] / 1000
df["battery_wh"] = df["battery_capacity_whr"]
df["density_wh_kg"] = (df["battery_density"] * 1000).round(1)      # Wh/g -> Wh/kg

# ---- 2. Segment ---------------------------------------------------------
# Khớp đúng định nghĩa H5: Gaming vs mọi laptop còn lại (gồm workstation)
df["segment"] = np.where(df["is_gaming_laptop"] == True, "Gaming", "Office")
df["segment_detail"] = np.where(df["is_gaming_laptop"] == True, "Gaming",
                        np.where(df["is_workstation"] == True, "Workstation", "Office"))

# ---- 3. Comparison-to-View Ratio ---------------------------------------
MIN_PV = 100
df["compare_view_ratio"] = (df["compare_count"] / df["pageview_count"].replace(0, np.nan)).round(4)
df["ratio_eligible"] = df["pageview_count"] >= MIN_PV

# ---- 4. Percentile ------------------------------------------------------
df["interest_percentile"] = df["interest_score"].rank(pct=True).round(3)
df["density_percentile"] = df["battery_density"].rank(pct=True).round(3)
df["density_pct_within_segment"] = (
    df.groupby("segment")["battery_density"].rank(pct=True).round(3))

In [9]:
# ---- 5. Golden group (H1') — tên nghiệp vụ ------------------------------
df["neg_weight"] = -df["laptop_weight_g"]
raw = percentile_quadrant(df, "battery_density", "neg_weight", 0.75, 0.25)
GOLD_MAP = {"top_x_top_y": "Lightweight + High Density",
            "top_x_bottom_y": "Heavy + High Density",
            "bottom_x_top_y": "Lightweight + Low Density",
            "bottom_x_bottom_y": "Heavy + Low Density",
            "middle": "Middle 50%"}
df["golden_group"] = raw.map(GOLD_MAP)
df.loc[df["battery_density"].isna() | df["laptop_weight_g"].isna(), "golden_group"] = "No data"
GOLD_ORDER = {"Lightweight + High Density": 1, "Heavy + High Density": 2, "Middle 50%": 3,
              "Lightweight + Low Density": 4, "Heavy + Low Density": 5, "No data": 6}
df["golden_group_order"] = df["golden_group"].map(GOLD_ORDER)

In [10]:
# ---- 6. H9' quadrant — tên nghiệp vụ ------------------------------------
raw9 = percentile_quadrant(df, "interest_score", "battery_density", 0.75, 0.25)
H9_MAP = {"top_x_top_y": "Star: High interest + High density",
          "top_x_bottom_y": "Content warning: High interest + Low density",
          "bottom_x_top_y": "Discovery opportunity: Low interest + High density",
          "bottom_x_bottom_y": "Low priority: Low interest + Low density",
          "middle": "Middle"}
df["h9_quadrant"] = raw9.map(H9_MAP)
df.loc[df["battery_density"].isna(), "h9_quadrant"] = "No data"
H9_ORDER = {"Star: High interest + High density": 1,
            "Content warning: High interest + Low density": 2,
            "Discovery opportunity: Low interest + High density": 3,
            "Low priority: Low interest + Low density": 4, "Middle": 5, "No data": 6}
df["h9_quadrant_order"] = df["h9_quadrant"].map(H9_ORDER)
 
def opp(row):
    if row["h9_quadrant"].startswith("Content warning"):
        return "High Interest / Low Density"
    if row["h9_quadrant"].startswith("Discovery"):
        return "Low Interest / High Density"
    return ""
df["opportunity_type"] = df.apply(opp, axis=1)
 
# Cảnh báo có phải chỉ do xu hướng chung của gaming? (thay cho is_confirmed_outlier)
df["warning_signal"] = np.where(
    df["opportunity_type"] == "High Interest / Low Density",
    np.where(df["density_pct_within_segment"] <= 0.25,
             "Low even within its segment", "Explained by segment trend"), "")

In [11]:
# ---- 7. Bucket Interest Score ------------------------------------------
bins = [-1, 500, 1000, 2000, 4000, np.inf]
labels = ["0-500", "500-1K", "1K-2K", "2K-4K", "4K+"]
df["interest_bucket"] = pd.cut(df["interest_score"], bins=bins, labels=labels)
df["interest_bucket_order"] = df["interest_bucket"].cat.codes + 1
df["interest_bucket"] = df["interest_bucket"].astype(str)
 
# ---- 8. Cột phụ đánh dấu trùng tên ----------------------------
df["name_is_duplicated"] = df["name"].duplicated(keep=False)
 
keep = ["id", "name", "name_is_duplicated", "segment", "segment_detail", "year_introduce",
        "pageview_count", "compare_count", "interest_score",
        "interest_bucket", "interest_bucket_order", "interest_percentile",
        "weight_kg", "battery_wh", "density_wh_kg", "density_percentile", "density_pct_within_segment",
        "compare_view_ratio", "ratio_eligible",
        "golden_group", "golden_group_order",
        "h9_quadrant", "h9_quadrant_order", "opportunity_type", "warning_signal"]
df_fact = df[keep].copy()
df_fact.to_csv("../data/processed/dashboard_laptop.csv", index=False)
print(df_fact.shape)

(133, 25)


In [12]:
# %% CELL 2 - Thong ke kiem dinh, nguong percentile, co cau Interest Score
g = df[df["segment"] == "Gaming"]; o = df[df["segment"] == "Office"]
rows = []
for tid, metric, col, scale, unit in [("H5", "Interest Score per laptop", "interest_score", 1, "points"),
                                       ("H1p", "Battery density", "battery_density", 1000, "Wh/kg")]:
    r = mann_whitney_with_effect_size(g[col], o[col])
    rows.append({"test_id": tid, "metric": metric, "unit": unit,
                 "group1": "Gaming", "n1": r["n1"], "median1": round(r["median1"] * scale, 1),
                 "group2": "Office", "n2": r["n2"], "median2": round(r["median2"] * scale, 1),
                 "p_value": float(f"{r['p_value']:.4g}"), "effect_size_r": r["effect_size_r"],
                 "test": "Mann-Whitney U (two-sided), effect size = rank-biserial r"})
df_tests = pd.DataFrame(rows)
df_tests.to_csv("../data/processed/dashboard_stat_tests.csv", index=False)
 
q = lambda s, p: float(s.quantile(p))
th = [("p25_interest", q(df["interest_score"], .25), "points"),
      ("p75_interest", q(df["interest_score"], .75), "points"),
      ("p25_density", q(df["density_wh_kg"], .25), "Wh/kg"),
      ("p75_density", q(df["density_wh_kg"], .75), "Wh/kg"),
      ("p25_weight", q(df["weight_kg"], .25), "kg"),
      ("p75_weight", q(df["weight_kg"], .75), "kg"),
      ("n_laptops", len(df), "count"),
      ("n_with_density_and_weight", int(df[["density_wh_kg", "weight_kg"]].notna().all(axis=1).sum()), "count"),
      ("min_pageview_for_ratio", 100, "pageviews")]
df_th = pd.DataFrame(th, columns=["metric", "value", "unit"])
df_th["value"] = df_th["value"].round(3)
df_th["as_of"] = datetime.now().strftime("%Y-%m-%d")
df_th.to_csv("../data/processed/dashboard_thresholds.csv", index=False)
print(df_tests.to_string()); print(df_th.to_string())
 
# Cơ cấu Interest Score
comp = pd.DataFrame({
    "action": ["Detail page view", "Comparison select"],
    "event_count": [int(df["pageview_count"].sum()), int(df["compare_count"].sum())],
    "weight": [1, 2]})
comp["contribution"] = comp["event_count"] * comp["weight"]
comp["share"] = (comp["contribution"] / comp["contribution"].sum()).round(4)
comp.to_csv("../data/processed/dashboard_interest_composition.csv", index=False)
print(comp)

  test_id                     metric    unit  group1  n1  median1  group2  n2  median2       p_value  effect_size_r                                                       test
0      H5  Interest Score per laptop  points  Gaming  47    643.0  Office  86    990.0  1.103000e-02         0.2674  Mann-Whitney U (two-sided), effect size = rank-biserial r
1     H1p            Battery density   Wh/kg  Gaming  47     36.9  Office  80     47.7  3.889000e-08         0.5856  Mann-Whitney U (two-sided), effect size = rank-biserial r
                      metric     value       unit       as_of
0               p25_interest   545.000     points  2026-10-17
1               p75_interest  1612.000     points  2026-10-17
2                p25_density    36.050      Wh/kg  2026-10-17
3                p75_density    50.000      Wh/kg  2026-10-17
4                 p25_weight     1.300         kg  2026-10-17
5                 p75_weight     2.026         kg  2026-10-17
6                  n_laptops   133.000   

In [13]:
# %% CELL 3 - Discovery path (dung DUNG cac cau SQL o cell "Trang 3" de trung cache data/raw)
search_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking WHERE event_name = 'search_for_device'",
    cache_dir="../data/raw")["session_id"])
detail_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'",
    cache_dir="../data/raw")["session_id"])
browse_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'load_more_device_home' "
    "   OR (event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'Home')",
    cache_dir="../data/raw")["session_id"])
 
search_led = detail_set & search_set
no_search = detail_set - search_set
browse_confirmed = no_search & browse_set
unknown = no_search - browse_set
assert len(search_led) + len(browse_confirmed) + len(unknown) == len(detail_set)
 
df_path = pd.DataFrame({
    "path": ["Search-led", "Browse-confirmed (Home / load-more)", "Unknown source"],
    "path_order": [1, 2, 3],
    "sessions": [len(search_led), len(browse_confirmed), len(unknown)]})
df_path["share_of_detail_sessions"] = (df_path["sessions"] / len(detail_set)).round(4)
df_path.to_csv("../data/processed/dashboard_discovery_path.csv", index=False)
print(df_path)
print(f"Detail sessions khong qua search: {len(no_search)/len(detail_set):.1%} | "
      f"trong do browse-confirmed: {len(browse_confirmed)/len(no_search):.1%}")

                                  path  path_order  sessions  \
0                           Search-led           1     28220   
1  Browse-confirmed (Home / load-more)           2      8836   
2                       Unknown source           3      7558   

   share_of_detail_sessions  
0                    0.6325  
1                    0.1981  
2                    0.1694  
Detail sessions khong qua search: 36.7% | trong do browse-confirmed: 53.9%


In [14]:
# %% CELL 4 - Validation (phai OK het truoc khi mo Power BI)
f = pd.read_csv("../data/processed/dashboard_laptop.csv")
checks = {
 "133 dòng, id duy nhất": len(f) == 133 and f["id"].is_unique,
 "density Wh/kg trong 5-90": f["density_wh_kg"].dropna().between(5, 90).all(),
 "weight kg trong 0.5-5": f["weight_kg"].dropna().between(0.5, 5).all(),
 "tổng interest = 162346 (khớp analysis_base)": f["interest_score"].sum() == pd.read_csv("../data/processed/analysis_base.csv")["interest_score"].sum(),
 "Golden = 24": (f["golden_group"] == "Lightweight + High Density").sum() == 24,
 "Gaming = 47, Office = 86": f["segment"].value_counts().to_dict() == {"Office": 86, "Gaming": 47},
 "không còn nhãn top_x/bottom_x": not f[["golden_group", "h9_quadrant"]].apply(lambda c: c.str.contains("_x_")).any().any(),
}
for k, v in checks.items(): print("OK  " if v else "FAIL", k)
print(f["h9_quadrant"].value_counts().to_string())
print(f["golden_group"].value_counts().to_string())
print(f["warning_signal"].value_counts().to_string())
print(f[f["opportunity_type"] == "High Interest / Low Density"][["name","interest_score","density_wh_kg","density_pct_within_segment","warning_signal"]].to_string())
print(f[f["opportunity_type"] == "Low Interest / High Density"][["name","interest_score","density_wh_kg"]].to_string())
print(f["interest_bucket"].value_counts().to_string()); print("ratio eligible:", f["ratio_eligible"].sum(), " ratio>1:", (f["compare_view_ratio"]>1).sum())

OK   133 dòng, id duy nhất
OK   density Wh/kg trong 5-90
OK   weight kg trong 0.5-5
OK   tổng interest = 162346 (khớp analysis_base)
OK   Golden = 24
OK   Gaming = 47, Office = 86
OK   không còn nhãn top_x/bottom_x
h9_quadrant
Middle                                                100
Low priority: Low interest + Low density                8
Discovery opportunity: Low interest + High density      8
Star: High interest + High density                      6
No data                                                 6
Content warning: High interest + Low density            5
golden_group
Middle 50%                    84
Lightweight + High Density    24
Heavy + Low Density           18
No data                        6
Lightweight + Low Density      1
warning_signal
Explained by segment trend     3
Low even within its segment    2
                                    name  interest_score  density_wh_kg  density_pct_within_segment               warning_signal
11                     Lenovo LOQ 15I